In [1]:
import os
import sys

# Add project root to path so we can import src modules
project_root = os.path.dirname(os.path.abspath(os.getcwd()))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

import pandas as pd
import numpy as np
import networkx as nx
from types import SimpleNamespace
from src.data.graph_strategies.strategy_factory import GraphStrategyFactory

In [2]:
data_path = "data/raw"

# Load auxiliary files (once)
games = pd.read_csv(os.path.join(data_path, "games.csv"))
players = pd.read_csv(os.path.join(data_path, "players.csv"))
plays = pd.read_csv(os.path.join(data_path, "plays.csv"))

# Load all tracking weeks, keeping only SNAP frames to save memory
tracking_frames = []
for week in range(1, 10):
    print(f"Loading week {week}...")
    week_data = pd.read_csv(os.path.join(data_path, f"tracking_week_{week}.csv"))
    snap_only = week_data[week_data["frameType"] == "SNAP"]
    tracking_frames.append(snap_only)

tracking_snap = pd.concat(tracking_frames, ignore_index=True)
del tracking_frames  # free memory
print(f"Total SNAP rows loaded: {len(tracking_snap)}")

Loading week 1...
Loading week 2...
Loading week 3...
Loading week 4...
Loading week 5...
Loading week 6...
Loading week 7...
Loading week 8...
Loading week 9...
Total SNAP rows loaded: 370852


In [3]:
# Filter valid plays only (remove spikes, kneels, sneaks, scrambles)
def get_play_result(play):
    if not pd.isna(play.get("qbSpike")) and play["qbSpike"]:
        return None
    if not pd.isna(play.get("qbKneel")) and play["qbKneel"]:
        return None
    if not pd.isna(play.get("qbSneak")) and play["qbSneak"]:
        return None
    if play.get("passResult") == "R":
        return None
    if not pd.isna(play.get("rushLocationType")):
        return 0
    if not pd.isna(play.get("passLocationType")):
        return 1
    if not pd.isna(play.get("passResult")):
        return 1
    return None

plays["playResult"] = plays.apply(get_play_result, axis=1)
valid_plays = plays.dropna(subset=["playResult"])
valid_keys = set(zip(valid_plays["gameId"], valid_plays["playId"]))
print(f"Valid plays: {len(valid_keys)}")

# Merge tracking with players to drop football rows (football has NaN nflId)
# Convert height/weight to metric as the pipeline does
players_merged = players.copy()
players_merged["height"] = players_merged["height"].str.split("-").apply(
    lambda x: round((int(x[0]) * 12 + int(x[1])) * 2.54)
)
players_merged["weight"] = round(players_merged["weight"] * 0.45359237)

tracking_snap = pd.merge(
    tracking_snap,
    players_merged[["nflId", "height", "weight", "position"]],
    on="nflId",
)

# Filter to valid plays only
tracking_snap["_key"] = list(zip(tracking_snap["gameId"], tracking_snap["playId"]))
tracking_snap = tracking_snap[tracking_snap["_key"].isin(valid_keys)].drop(columns=["_key"])
print(f"Tracking rows after filtering: {len(tracking_snap)}")

Valid plays: 15416
Tracking rows after filtering: 339152


In [4]:
STRATEGIES = ["CLOSEST-", "QB-CLOSEST-", "DELAUNAY", "GABRIEL", "RNG", "MST"]

def count_edges_from_connections(connections_dict):
    """Count unique undirected edges per play from a connections dict."""
    edge_counts = []
    for game_id, game_data in connections_dict.items():
        for play_id, play_data in game_data.items():
            if "connections" not in play_data or not play_data["connections"]:
                continue
            edges = set()
            for player_id, conns in play_data["connections"].items():
                for conn in conns:
                    edge = tuple(sorted([player_id, conn["nflId"]]))
                    edges.add(edge)
            edge_counts.append(len(edges))
    return edge_counts

results = {}
for strategy_name in STRATEGIES:
    print(f"\nProcessing strategy: {strategy_name}")
    config = SimpleNamespace(
        EDGE_STRATEGY=strategy_name,
        N=2,
        QB_LINK=False,
        DOWN_SAMPLE=False,
        FORCE_REBUILD=True,
    )

    strategy = GraphStrategyFactory.create_strategy(config)

    # Pass original players (with string 'position') for QB-CLOSEST
    connections_dict = strategy.calculate_connections(tracking_snap.copy(), players.copy())

    edge_counts = count_edges_from_connections(connections_dict)

    results[strategy_name] = {
        "mean": np.mean(edge_counts),
        "std": np.std(edge_counts),
        "min": np.min(edge_counts),
        "max": np.max(edge_counts),
        "median": np.median(edge_counts),
        "total_plays": len(edge_counts),
    }
    print(f"  {strategy_name}: mean={results[strategy_name]['mean']:.2f} ± {results[strategy_name]['std']:.2f}, "
          f"min={results[strategy_name]['min']}, max={results[strategy_name]['max']}, "
          f"plays={results[strategy_name]['total_plays']}")


Processing strategy: CLOSEST-
[INFO] 2026-05-04 18:55:43 - Calculating connections using 2 closest players strategy...
  CLOSEST-: mean=31.99 ± 1.54, min=25, max=37, plays=15416

Processing strategy: QB-CLOSEST-
[INFO] 2026-05-04 18:56:16 - Calculating connections using 2 closest players strategy...
  QB-CLOSEST-: mean=50.56 ± 2.05, min=30, max=71, plays=15416

Processing strategy: DELAUNAY
[INFO] 2026-05-04 18:59:15 - Calculating connections using Delaunay triangulation...
  DELAUNAY: mean=55.94 ± 0.88, min=52, max=59, plays=15416

Processing strategy: GABRIEL
[INFO] 2026-05-04 18:59:54 - Calculating connections using Gabriel Graph...
  GABRIEL: mean=37.47 ± 2.38, min=26, max=47, plays=15416

Processing strategy: RNG
[INFO] 2026-05-04 19:01:04 - Calculating connections using Relative Neighborhood Graph...
  RNG: mean=23.72 ± 1.21, min=21, max=28, plays=15416

Processing strategy: MST
[INFO] 2026-05-04 19:02:36 - Calculating connections using Minimum Spanning Tree...
  MST: mean=21.00

In [5]:
# Build results DataFrame
df = pd.DataFrame(results).T
df.index.name = "Strategy"
df = df.sort_values("mean")
print(df.to_string())

# Generate LaTeX table
strategy_labels = {
    "MST": "MST",
    "RNG": "RNG",
    "CLOSEST-": "$k$-NNG ($k$=2)",
    "GABRIEL": "Gabriel",
    "QB-CLOSEST-": "QB-Centric $k$-NNG",
    "DELAUNAY": "Delaunay",
}

print("\n\n% ---- LaTeX Table ----")
print(r"\begin{table}[h!]")
print(r"    \centering")
print(r"    \caption{Mean number of edges per graph construction strategy across all valid plays.}")
print(r"    \label{tab:edge_counts}")
print(r"    \begin{tabular}{l c c c c}")
print(r"        \toprule")
print(r"        \textbf{Strategy} & \textbf{Mean Edges} & \textbf{Std} & \textbf{Min} & \textbf{Max} \\")
print(r"        \midrule")

for strategy_name in df.index:
    r = results[strategy_name]
    label = strategy_labels.get(strategy_name, strategy_name)
    print(f"        {label} & {r['mean']:.2f} & {r['std']:.2f} & {r['min']} & {r['max']} \\\\")

print(r"        \bottomrule")
print(r"    \end{tabular}")
print(r"\end{table}")

                  mean       std   min   max  median  total_plays
Strategy                                                         
MST          21.000000  0.000000  21.0  21.0    21.0      15416.0
RNG          23.716528  1.212909  21.0  28.0    24.0      15416.0
CLOSEST-     31.993254  1.536262  25.0  37.0    32.0      15416.0
GABRIEL      37.474053  2.377262  26.0  47.0    38.0      15416.0
QB-CLOSEST-  50.557602  2.045551  30.0  71.0    51.0      15416.0
DELAUNAY     55.943825  0.878569  52.0  59.0    56.0      15416.0


% ---- LaTeX Table ----
\begin{table}[h!]
    \centering
    \caption{Mean number of edges per graph construction strategy across all valid plays.}
    \label{tab:edge_counts}
    \begin{tabular}{l c c c c}
        \toprule
        \textbf{Strategy} & \textbf{Mean Edges} & \textbf{Std} & \textbf{Min} & \textbf{Max} \\
        \midrule
        MST & 21.00 & 0.00 & 21 & 21 \\
        RNG & 23.72 & 1.21 & 21 & 28 \\
        $k$-NNG ($k$=2) & 31.99 & 1.54 & 25 & 37 \\
 